[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/09_metricas/09_proyecto_metricas.ipynb)

# Proyecto 09 · Métricas de negocio y de modelo de SupportOps: histogramas, coste y exemplars

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 30 min |
| **Servicios de Compose** | **Opcionales**: `core` + `metrics` (`docker compose --profile core --profile metrics up -d`, ≈ 0,45 GB). La rúbrica base corre **sin Docker**; con stack se añaden comprobaciones en vivo contra Prometheus |
| **Prerrequisitos** | Lección `09_metricas` (instrumentos, buckets, cardinalidad, exemplars, PromQL) y `setup_telemetry`/`OTelInstrumentor` del módulo 03 |
| **Conexión** | Construyes tú la capa de métricas que en la lección te dimos hecha (`MetricsInstrumentor`). Es la base de los paneles del módulo 10 y del FinOps del 13 |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El equipo de finanzas pregunta *«¿cuánto nos cuesta cada ticket y qué modelo lo gasta?»*; el on-call, *«¿cuál es el p95 de hoy y qué herramienta lo está empeorando?»*; y el responsable de plataforma, *«¿cuántas series va a añadir esto a Prometheus?»*. Las trazas no responden a ninguna de las tres a escala: necesitas **métricas** bien diseñadas: coste y tokens como contadores, latencias como histogramas con buckets que midan **de verdad**, atributos acotados y *exemplars* para saltar a la traza.

## Datos

Tráfico sintético de SupportOps con **semilla fija** (`generate_tickets(…, seed=42)`, `LLM_PROVIDER=mock`, tiempos simulados deterministas): sin descargas ni licencias. Cadena de datos: **stack local** (opcional, para la parte en vivo) → **modo degradado** (todo en memoria con `InMemoryMetricReader`, mismo esquema) → datos regenerables por semilla. No hay datasets externos.

## Qué vas a construir (los 6 TODOs)

| TODO | Entregable |
|---|---|
| 1 | `PRICES` y `cost()`: tabla de precios **ilustrativa** y función de coste (error claro con modelos desconocidos) |
| 2 | `LATENCY_BOUNDS` y `TOKEN_BOUNDS`: límites de bucket que midan p50/p95/p99 con poco error (se evalúa con **otras semillas**) |
| 3 | `build_metrics(meter)`: los 8 instrumentos con nombre, tipo y unidad exactos |
| 4 | `Recorder`: un `Instrumentor` que registra las métricas (RED + tokens + coste) con atributos **acotados** |
| 5 | `series_budget()`: presupuesto de series (∏ valores × (B+3) en histogramas) |
| 6 | `PROMQL`: seis consultas del *playbook* (con los nombres que producirá Prometheus) |

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, `mock`, semillas 7 y 99) |
|---|---|---|---|
| R1 | `cost()` | 5 casos exactos (tolerancia 1e-9) y `KeyError` en modelo desconocido | 5/5 + error |
| R2 | Buckets | ≤ 20 límites crecientes, con **3,0 s** (umbral de SLO) entre ellos; error de p50/p95/p99 de latencia ≤ 3 % / 10 % / 20 % y de p50/p95 de tokens ≤ 3 % / 10 % en las semillas **7 y 99** (1 000 tickets) | La solución mide ≤ 1 % / 7 % / 16 % y ≤ 0,3 % / 7,1 %; los límites de la spec dan 6-7 % (p50) y 59 % (tokens) |
| R3 | Instrumentos | Nombres, tipos y unidades exactos; los histogramas usan **tus** límites | 8/8 |
| R4 | Conciliación | Peticiones = 200 y por estado; tokens y coste = verdad del agente (rtol 1e-4; el coste 2e-3 porque el agente redondea `cost_usd` a 6 decimales); cuenta y suma de duraciones; nº de llamadas LLM y de herramientas | 8 magnitudes |
| R5 | Cardinalidad | Atributos **exactamente** los pedidos; sin ids; series totales ≤ presupuesto declarado y ≤ 700; `series_budget` correcto | ver celda |
| R6 | Exemplars | ≥ 1 exemplar en la duración y el 100 % apunta a una traza capturada | 100 % |
| R7 | PromQL (estático) | 6 consultas con la forma correcta (`sum by (le)`, ventana ≥ 1 m, nombres válidos…) | 6/6 |
| L1-L3 | *(con stack)* | Las consultas devuelven datos y se parecen a la verdad (tolerancias amplias por el desfase del *scrape*) | informativo |

> 📏 Los umbrales salen de la solución de referencia (abajo), probada con semillas 42, 7, 99 y 123. No dependen de tu máquina: los tiempos son simulados.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import json, math, time, collections, logging
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, Incident, ...
from supportops import viz, sample_data, backends
from supportops.telemetry import setup_telemetry, OTelInstrumentor
from supportops.otel_metrics import *                    # MetricsInstrumentor, PriceTable, run_live, prom_instant, prom_range, ...
viz.apply_style()
pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 90)
SEED = 42
FAST_DEV_RUN = True   # True: tráfico en vivo corto (~100 s) y pocos tickets. False: 4 min de tráfico y más tickets

from contextlib import contextmanager
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded
print("Modo:", "COMPLETO (con Prometheus)" if LIVE else "DEGRADADO (rúbrica base en memoria; sin comprobaciones en vivo)")

## Paso 0 · Herramientas que te damos hechas

Son las del lección: la traducción de nombres a Prometheus, `histogram_quantile` y una forma de medir el error de un percentil con unos límites.

In [ ]:
import re
UNIT_WORDS = {"s": "seconds", "ms": "milliseconds", "By": "bytes", "%": "percent"}
def prom_name(name, unit="", kind="sum", monotonic=True):
    """Serie(s) de Prometheus de una métrica OTel (misma función que en la lección)."""
    base = re.sub(r"_+", "_", re.sub(r"[^a-zA-Z0-9_:]", "_", name))
    u = re.sub(r"\{[^}]*\}", "", unit or "").strip()
    word = UNIT_WORDS.get(u, re.sub(r"[^a-zA-Z0-9_]", "_", u)) if u and u != "1" else ""
    if kind == "gauge" and u == "1": word = "ratio"
    if word and not base.endswith("_" + word): base += "_" + word
    if kind == "sum" and monotonic and not base.endswith("_total"): base += "_total"
    return [base + s for s in ("_bucket", "_sum", "_count")] if kind == "histogram" else [base]

def to_buckets(values, bounds):
    idx = np.searchsorted(bounds, np.asarray(values), side="left"); counts = np.bincount(idx, minlength=len(bounds) + 1)
    return list(zip([*bounds, float("inf")], np.cumsum(counts)))

def hist_quantile(q, buckets):
    total = buckets[-1][1]; rank = q * total
    b = next(i for i, (_, c) in enumerate(buckets) if c >= rank)
    if b == len(buckets) - 1: return buckets[-2][0]
    lo, c_lo = (0.0, 0.0) if b == 0 else buckets[b - 1]; hi, c_hi = buckets[b]
    return lo + (hi - lo) * (rank - c_lo) / (c_hi - c_lo)

def bucket_quantile(values, bounds, q): return hist_quantile(q, to_buckets(values, bounds))

def sample_runs(seed, n=1000):
    """Latencias del agente y tokens de entrada por llamada de n tickets (para evaluar buckets)."""
    tk = generate_tickets(n, seed=seed, fault_rates=DEFAULT_FAULT_RATES); res = run_tickets(SupportAgent(), tk)
    return (np.array([r.latency_s for r in res]), np.array([s.input_tokens for r in res for s in r.steps if s.kind == "llm" and s.status == "ok"]))

def quantile_errors(values, bounds, qs=(0.5, 0.95, 0.99)):
    """Error relativo (%) de histogram_quantile con esos límites frente al percentil verdadero."""
    return {f"p{int(q*100)}": round(100 * (bucket_quantile(values, bounds, q) - np.quantile(values, q)) / np.quantile(values, q), 1) for q in qs}

TRAIN_LAT, TRAIN_TOK = sample_runs(42, 600)                 # muestra de «entrenamiento» (la rúbrica evalúa con OTRAS semillas)
print(f"Muestra de entrenamiento: latencia p50={np.quantile(TRAIN_LAT, .5):.2f}s p95={np.quantile(TRAIN_LAT, .95):.2f}s p99={np.quantile(TRAIN_LAT, .99):.2f}s · tokens de entrada p50={np.quantile(TRAIN_TOK, .5):.0f} p95={np.quantile(TRAIN_TOK, .95):.0f}")
print("Con los límites de la spec GenAI (para comparar):", quantile_errors(TRAIN_LAT, SPEC_AGENT_DURATION_BOUNDARIES), quantile_errors(TRAIN_TOK, SPEC_TOKEN_BOUNDARIES, (0.5, 0.95)))

## Paso 1 · Del token al dólar (TODO 1)

Define `PRICES` con los precios **ilustrativos** del curso (**no son precios reales**): `mock-small` = 0,25 USD de entrada y 1,25 de salida por millón de tokens; `mock-large` = 3,00 y 15,00. Implementa `cost(model, input_tokens, output_tokens)` en USD. Un modelo desconocido debe lanzar `KeyError` (mejor un error claro que un coste 0 en silencio).

<details>
<summary>🪜 Pista 1</summary>

$C = (n_{in}\,p_{in} + n_{out}\,p_{out})/10^6$. Guarda `{modelo: (p_in, p_out)}` y desempaqueta con `pin, pout = PRICES[model]` (el `KeyError` sale solo).

</details>

In [ ]:
PRICES = {}   # TODO 1a: {"mock-small": (0.25, 1.25), "mock-large": (3.00, 15.00)}  (USD por millón de tokens: entrada, salida)

def cost(model: str, input_tokens: int, output_tokens: int) -> float:
    # TODO 1b
    raise NotImplementedError("TODO 1")

try:
    print(f"{cost('mock-small', 1000, 100):.6f} USD (esperado 0.000375)")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Buckets que midan de verdad (TODO 2)

Elige `LATENCY_BOUNDS` (segundos) y `TOKEN_BOUNDS` (tokens de entrada por llamada). Restricciones: **≤ 20 límites**, estrictamente crecientes, y `LATENCY_BOUNDS` debe incluir **3.0** (tu umbral de SLO). Mira la distribución de la muestra de entrenamiento y pon los límites **donde está la masa**. La rúbrica los evalúa con **otras semillas** (7 y 99): no sobreajustes a la 42.

<details>
<summary>🪜 Pista 1</summary>

Mira `np.quantile(TRAIN_LAT, [.05,.25,.5,.75,.9,.95,.99])`. Quieres rejilla fina (cada 0,25–0,5 s) entre ≈ 0,25 s y 5 s y pocos límites en la cola (7,5; 10; 20; 40). Para los tokens, la masa está entre ≈ 300 y 900: límites cada 50-100 ahí.

</details>

<details>
<summary>🪜 Pista 2</summary>

Una disposición que funciona: latencia `[0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5, 7.5, 10, 20, 40]`; tokens `[100, 200, 300, 350, 400, 450, 500, 600, 800, 1000, 1500, 2000, 4000, 8000]`. Pero entiende *por qué*: comprueba tú el error con `quantile_errors`.

</details>

In [ ]:
LATENCY_BOUNDS = []   # TODO 2a (segundos)
TOKEN_BOUNDS = []     # TODO 2b (tokens)

if LATENCY_BOUNDS and TOKEN_BOUNDS:
    print("latencia:", quantile_errors(TRAIN_LAT, LATENCY_BOUNDS), "· tokens:", quantile_errors(TRAIN_TOK, TOKEN_BOUNDS, (0.5, 0.95)))
else:
    print("Pendiente: rellena LATENCY_BOUNDS y TOKEN_BOUNDS")

## Paso 3 · Los instrumentos (TODO 3)

`build_metrics(meter)` devuelve un `dict` con **estos ocho instrumentos exactamente** (nombre OTel, tipo, unidad). Los histogramas llevan sus límites como *advisory* (`explicit_bucket_boundaries_advisory=`):

| clave | nombre OTel | tipo | unidad | límites |
|---|---|---|---|---|
| `requests` | `proj09.agent.requests` | Counter | `{request}` | — |
| `duration` | `proj09.agent.duration` | Histogram | `s` | `LATENCY_BOUNDS` |
| `in_tokens` | `proj09.llm.input_tokens` | Counter | `{token}` | — |
| `out_tokens` | `proj09.llm.output_tokens` | Counter | `{token}` | — |
| `cost` | `proj09.llm.cost` | Counter | `USD` | — |
| `llm_duration` | `proj09.llm.call.duration` | Histogram | `s` | `LATENCY_BOUNDS` |
| `tool_duration` | `proj09.tool.duration` | Histogram | `s` | `LATENCY_BOUNDS` |
| `tokens_per_call` | `proj09.llm.input_tokens_per_call` | Histogram | `{token}` | `TOKEN_BOUNDS` |

<details>
<summary>🪜 Pista 1</summary>

`meter.create_counter(name, unit=..., description=...)` y `meter.create_histogram(name, unit=..., description=..., explicit_bucket_boundaries_advisory=[...])`. Devuelve un diccionario `{"requests": ..., ...}`.

</details>

In [ ]:
def build_metrics(meter) -> dict:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    _mp, _rd = make_meter_provider(otlp_endpoint=None); print(sorted(build_metrics(_mp.get_meter("t"))))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · El instrumentor de métricas (TODO 4)

`Recorder` es un `Instrumentor` (módulo 01-03). Registra:

- **al terminar cada ticket** (`agent_run`, leyendo `ctx.result`): `requests` con atributos **`status`, `intent`, `tier`** (`intent` = `r.intent_pred` o `"unknown"`; `tier` = `ctx.ticket.tier`) y `duration` (`r.latency_s`) con atributos **`model`, `status`**;
- **en cada llamada al LLM** (`llm_call`): `llm_duration` (duración con el reloj simulado `ctx.clock.now_s()`) con **`model`, `outcome`** (`ok`/`error`); si hubo respuesta: `in_tokens`, `out_tokens`, `cost` (con `cost()`) y `tokens_per_call` (tokens de entrada), todos con **`model`**;
- **en cada herramienta** (`tool_call`): `tool_duration` con **`tool`** (`info.call.name`) y **`outcome`**.

Las llamadas **fallidas también cuentan** (duración con `outcome="error"`, sin tokens). **Ningún** atributo puede ser un id (`ticket_id`, `customer_id`…).

<details>
<summary>🪜 Pista 1</summary>

Sigue el patrón de la lección: `try: yield / except BaseException: outcome = "error"; raise / finally: <registrar>`. En `llm_call` la respuesta está en `info.response` (puede ser `None` si hubo error): `info.response.usage.input_tokens`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Dentro de `finally` de `llm_call`: `dur = ctx.clock.now_s() - t0`; `self.m["llm_duration"].record(dur, {"model": info.model, "outcome": outcome})`; `r = info.response`; `if r is not None:` suma tokens y coste con `{"model": info.model}`.

</details>

In [ ]:
class Recorder(Instrumentor):
    def __init__(self, meter_provider):
        self.m = build_metrics(meter_provider.get_meter("proj09"))

    @contextmanager
    def agent_run(self, ctx):
        try:
            yield
        finally:
            r = ctx.result
            if r is not None:
                pass   # TODO 4a: requests y duration

    @contextmanager
    def llm_call(self, ctx, info):
        t0 = ctx.clock.now_s(); outcome = "ok"
        try:
            yield
        except BaseException:
            outcome = "error"; raise
        finally:
            pass       # TODO 4b: llm_duration (+ tokens, coste y tokens_per_call si hay respuesta)

    @contextmanager
    def tool_call(self, ctx, info):
        t0 = ctx.clock.now_s(); outcome = "ok"
        try:
            yield
        except BaseException:
            outcome = "error"; raise
        finally:
            pass       # TODO 4c: tool_duration

In [ ]:
def run_inmemory(n=200, seed=SEED):
    """Ejecuta n tickets con trazas (para exemplars) + tu Recorder; devuelve filas de métricas, resultados y trazas capturadas."""
    tr = setup_telemetry("proj09-lab", mode="memory", metrics=False, logs=False)
    mp, rd = make_meter_provider(otlp_endpoint=None)
    ag = SupportAgent(instrumentor=CompositeInstrumentor(OTelInstrumentor(tr, emit_metrics=False), Recorder(mp)))
    tk = generate_tickets(n, seed=seed, fault_rates=DEFAULT_FAULT_RATES); res = run_tickets(ag, tk)
    return pd.DataFrame(metric_rows(rd)), res, {s["trace_id"] for s in tr.span_records()}

try:
    ROWS, RES, TRACE_IDS = run_inmemory()
    display(ROWS.groupby("name").agg(kind=("kind", "first"), unit=("unit", "first"), series=("name", "size")))
except NotImplementedError as e:
    ROWS = None; print("Pendiente:", e)

## Paso 5 · Presupuesto de series (TODO 5)

Implementa `series_budget(attrs, n_bounds=None)`: `attrs` es `{atributo: nº máximo de valores}`; devuelve el nº **máximo** de series que generaría el instrumento en Prometheus: el producto de los valores y, si es un histograma clásico (`n_bounds` = nº de límites finitos), por **`n_bounds + 3`** (`n_bounds+1` buckets incluido `+Inf`, más `_sum` y `_count`). Sin atributos → 1 serie.
Después rellena `DECLARED` con el presupuesto de **tus 8 instrumentos** según el diseño de arriba (valores máximos razonables: 5 estados, 5 intenciones, 2 tiers, 2 modelos, 4 herramientas, 2 *outcomes*) y calcula `TOTAL_BUDGET`.

<details>
<summary>🪜 Pista 1</summary>

`math.prod(attrs.values()) * (n_bounds + 3 if n_bounds is not None else 1)`. Ejemplo: `requests` = 5·5·2 = 50 series; `duration` = (2 modelos · 5 estados) · (B+3).

</details>

In [ ]:
import math
def series_budget(attrs: dict, n_bounds: int | None = None) -> int:
    # TODO 5a
    raise NotImplementedError("TODO 5")

DECLARED = {}      # TODO 5b: {"requests": ..., "duration": ..., "in_tokens": ..., "out_tokens": ..., "cost": ..., "llm_duration": ..., "tool_duration": ..., "tokens_per_call": ...}
TOTAL_BUDGET = 0   # TODO 5c: suma de DECLARED

## Paso 6 · El *playbook* de PromQL (TODO 6)

Escribe en `PROMQL` seis consultas (cadenas). Los nombres de serie de Prometheus los produce la traducción de la lección (`prom_name`): **no los adivines, calcúlalos**. Usa ventanas de **`[1m]`** (el *scrape* es de 15 s) y `sum by (le)` en los percentiles.

| clave | pregunta |
|---|---|
| `rate` | peticiones por segundo |
| `error_ratio` | fracción de peticiones con `status` `error` o `max_steps` |
| `p95` | p95 de la duración del agente |
| `cost_per_hour` | USD por hora |
| `tokens_per_request` | tokens de entrada por petición (cociente de dos tasas) |
| `slowest_tool` | herramienta con mayor duración **media** (`topk(1, …)` de `sum/count` por `tool`) |

<details>
<summary>🪜 Pista 1</summary>

Cada consulta tiene **una forma fija**: contador → `sum(rate(<serie>_total[1m]))`; cociente → una `sum(rate())` entre otra; percentil → `histogram_quantile(q, sum by (le) (rate(<serie>_bucket[1m])))`; media → `rate(..._sum)` entre `rate(..._count)`. Escribe primero la pregunta en palabras («¿qué dividido entre qué?») y luego busca el nombre exacto de cada serie imprimiendo `sorted(VALID)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Reglas de traducción (sección 4): los puntos pasan a `_`; el tipo añade sufijo (`_total` en contadores, `_bucket`/`_sum`/`_count` en histogramas); la unidad `s` se escribe `_seconds`; una unidad entre llaves como `{request}` o `{token}` **se descarta**, pero `USD` se **conserva** (`..._USD_total`). Para filtrar por estado: `{status=~"error|max_steps"}`. `topk(1, A / B)` necesita que A y B estén agregados `by (tool)`.

</details>

In [ ]:
VALID = set()
for _n, _u, _k, _m in [("proj09.agent.requests", "{request}", "sum", True), ("proj09.agent.duration", "s", "histogram", True), ("proj09.llm.input_tokens", "{token}", "sum", True),
                       ("proj09.llm.output_tokens", "{token}", "sum", True), ("proj09.llm.cost", "USD", "sum", True), ("proj09.llm.call.duration", "s", "histogram", True),
                       ("proj09.tool.duration", "s", "histogram", True), ("proj09.llm.input_tokens_per_call", "{token}", "histogram", True)]:
    VALID.update(prom_name(_n, _u, _k, _m))
print(len(VALID), "series válidas, p. ej.:", sorted(VALID)[:3])

PROMQL = {   # TODO 6: rellena las seis claves: rate, error_ratio, p95, cost_per_hour, tokens_per_request, slowest_tool
}

## Paso 7 (opcional, con stack) · Tráfico en vivo y validación en Prometheus

Si el stack está arriba, enviamos 60 s de tráfico con **tu** `Recorder` por el Collector y evaluamos **tus** consultas contra Prometheus. Es lo que hará la rúbrica en las filas `L1-L3`.

In [ ]:
def run_live_project(secs=60):
    tr = setup_telemetry("proj09-lab", metrics=False, logs=False)                    # trazas por OTLP (para exemplars)
    mp, rd = make_meter_provider("proj09-lab", otlp_endpoint="http://localhost:4318/v1/metrics", interval_s=5)
    ag = SupportAgent(instrumentor=CompositeInstrumentor(OTelInstrumentor(tr, emit_metrics=False), Recorder(mp)))
    tk = generate_tickets(secs * 2, seed=SEED, duration_s=secs, fault_rates=DEFAULT_FAULT_RATES, start_s=0.0)
    t0 = time.time(); res = run_live(ag, tk, secs); t1 = time.time()
    tr.force_flush(); mp.force_flush(); time.sleep(22)
    return res, t0, t1

LIVE_RUN = None
if LIVE and PROMQL and ROWS is not None:
    try: LIVE_RUN = run_live_project(60)
    except Exception as e: print("No se pudo ejecutar el tráfico en vivo:", type(e).__name__, str(e)[:100])
else:
    print("Se omite (sin stack, o aún faltan TODOs).")

## Paso 8 · Rúbrica

In [ ]:
def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": ("✅ OK" if ok else "❌ FALLA") if ok is not None else "⏭ n/a", "detalle": det})
    # R1 ---------------------------------------------------------------------------------------------------
    try:
        cases = [("mock-small", 1000, 100, 0.000375), ("mock-small", 0, 0, 0.0), ("mock-large", 2000, 300, 0.0105), ("mock-large", 1, 1, 0.000018), ("mock-small", 354, 19, 0.00011225)]
        good = sum(abs(cost(m, a, b) - e) < 1e-9 for m, a, b, e in cases)
        try: cost("gpt-desconocido", 1, 1); raised = False
        except KeyError: raised = True
        add("R1", "cost(): 5 casos exactos y KeyError en modelo desconocido", good == 5 and raised, f"{good}/5 · KeyError={raised}")
    except Exception as e: add("R1", "cost()", False, type(e).__name__)
    # R2 ---------------------------------------------------------------------------------------------------
    try:
        ok_shape = (0 < len(LATENCY_BOUNDS) <= 20 and 0 < len(TOKEN_BOUNDS) <= 20 and all(b > a for a, b in zip(LATENCY_BOUNDS, LATENCY_BOUNDS[1:]))
                    and all(b > a for a, b in zip(TOKEN_BOUNDS, TOKEN_BOUNDS[1:])) and 3.0 in LATENCY_BOUNDS)
        worst = {"p50": 0, "p95": 0, "p99": 0, "t50": 0, "t95": 0}
        for sd in (7, 99):
            lat, tok = sample_runs(sd, 1000); e = quantile_errors(lat, LATENCY_BOUNDS); t = quantile_errors(tok, TOKEN_BOUNDS, (.5, .95))
            for k in ("p50", "p95", "p99"): worst[k] = max(worst[k], abs(e[k]))
            worst["t50"], worst["t95"] = max(worst["t50"], abs(t["p50"])), max(worst["t95"], abs(t["p95"]))
        ok = ok_shape and worst["p50"] <= 3 and worst["p95"] <= 10 and worst["p99"] <= 20 and worst["t50"] <= 3 and worst["t95"] <= 10
        add("R2", "Buckets: forma válida, incluye 3.0 y errores de percentil acotados (semillas 7 y 99)", ok, f"forma={ok_shape} · peor error lat p50/p95/p99 = {worst['p50']}/{worst['p95']}/{worst['p99']} % · tokens p50/p95 = {worst['t50']}/{worst['t95']} %")
    except Exception as e: add("R2", "Buckets", False, type(e).__name__ + ": " + str(e)[:60])
    # R3 ---------------------------------------------------------------------------------------------------
    SPEC = {"requests": ("proj09.agent.requests", "{request}", "sum", None), "duration": ("proj09.agent.duration", "s", "histogram", "lat"),
            "in_tokens": ("proj09.llm.input_tokens", "{token}", "sum", None), "out_tokens": ("proj09.llm.output_tokens", "{token}", "sum", None),
            "cost": ("proj09.llm.cost", "USD", "sum", None), "llm_duration": ("proj09.llm.call.duration", "s", "histogram", "lat"),
            "tool_duration": ("proj09.tool.duration", "s", "histogram", "lat"), "tokens_per_call": ("proj09.llm.input_tokens_per_call", "{token}", "histogram", "tok")}
    try:
        mp, rd = make_meter_provider(otlp_endpoint=None); ins = build_metrics(mp.get_meter("r3")); good = []
        for k, (name, unit, kind, b) in SPEC.items():
            i = ins[k]; (i.record if kind == "histogram" else i.add)(1)
        got = {r["name"]: r for r in metric_rows(rd)}
        for k, (name, unit, kind, b) in SPEC.items():
            r = got.get(name); ok = r is not None and r["unit"] == unit and r["kind"] == ("histogram" if kind == "histogram" else "sum")
            if ok and b: ok = r["bounds"] == [float(x) for x in (LATENCY_BOUNDS if b == "lat" else TOKEN_BOUNDS)]
            good.append(ok)
        add("R3", "build_metrics(): 8 instrumentos con nombre, tipo, unidad y límites exactos", all(good), f"{sum(good)}/8")
    except Exception as e: add("R3", "build_metrics", False, type(e).__name__ + ": " + str(e)[:60])
    # R4 ---------------------------------------------------------------------------------------------------
    try:
        R, res, trace_ids = run_inmemory(200, SEED)
        def tot(name, **at):
            s = R[R.name == name]
            for k, v in at.items(): s = s[s.attributes.map(lambda a: a.get(k) == v)]
            return float(s["value"].sum()) if len(s) and s["kind"].iloc[0] == "sum" else float(s["count"].sum())
        truth = pd.DataFrame([(r.status, r.input_tokens, r.output_tokens, r.cost_usd, r.latency_s, r.n_llm_calls, r.n_tool_calls) for r in res], columns="status tin tout cost lat nl nt".split())
        checks = {"peticiones": (tot("proj09.agent.requests"), 200),
                  "peticiones «ok»": (tot("proj09.agent.requests", status="ok"), (truth.status == "ok").sum()),
                  "tokens entrada": (tot("proj09.llm.input_tokens"), truth.tin.sum()), "tokens salida": (tot("proj09.llm.output_tokens"), truth.tout.sum()),
                  "coste": (tot("proj09.llm.cost"), truth.cost.sum()), "Σ duración": (float(R[R.name == "proj09.agent.duration"]["sum"].sum()), truth.lat.sum()),
                  "llamadas LLM": (tot("proj09.llm.call.duration"), truth.nl.sum()), "herramientas": (tot("proj09.tool.duration"), truth.nt.sum())}
        bad = [k for k, (a, b) in checks.items() if not np.isclose(a, b, rtol=(2e-3 if k == "coste" else 1e-4), atol=1e-6)]
        add("R4", "Conciliación con la verdad del agente (8 magnitudes)", not bad, "todas coinciden" if not bad else f"no cuadran: {bad}")
    except Exception as e: R = None; add("R4", "Conciliación", False, type(e).__name__ + ": " + str(e)[:60])
    # R5 ---------------------------------------------------------------------------------------------------
    try:
        want = {"proj09.agent.requests": {"status", "intent", "tier"}, "proj09.agent.duration": {"model", "status"}, "proj09.llm.input_tokens": {"model"},
                "proj09.llm.output_tokens": {"model"}, "proj09.llm.cost": {"model"}, "proj09.llm.call.duration": {"model", "outcome"},
                "proj09.tool.duration": {"tool", "outcome"}, "proj09.llm.input_tokens_per_call": {"model"}}
        keys_ok = all({frozenset(a) for a in R[R.name == n].attributes} == {frozenset(w)} for n, w in want.items())
        pts = {n: len(R[R.name == n]) for n in want}
        def per_point(n):
            if R[R.name == n].iloc[0]["kind"] != "histogram": return 1
            return (len(TOKEN_BOUNDS) if "per_call" in n else len(LATENCY_BOUNDS)) + 3
        actual = sum(p * per_point(n) for n, p in pts.items())
        formula = (series_budget({"a": 2, "b": 3}) == 6 and series_budget({"a": 2, "b": 3}, 10) == 78 and series_budget({}) == 1)
        ok = keys_ok and formula and actual <= TOTAL_BUDGET <= 700 and TOTAL_BUDGET > 0 and len(DECLARED) == 8
        add("R5", "Cardinalidad: atributos exactos, series reales ≤ presupuesto declarado ≤ 700, series_budget correcto", ok, f"atributos={keys_ok} · fórmula={formula} · series reales={actual} · presupuesto declarado={TOTAL_BUDGET}")
    except Exception as e: add("R5", "Cardinalidad", False, type(e).__name__ + ": " + str(e)[:60])
    # R6 ---------------------------------------------------------------------------------------------------
    try:
        ex = [e for _, r in R[R.name == "proj09.agent.duration"].iterrows() for e in r["exemplars"]]
        ok = len(ex) >= 1 and all(e["trace_id"] in trace_ids for e in ex)
        add("R6", "Exemplars: ≥ 1 en la duración y todos apuntan a una traza capturada", ok, f"{len(ex)} exemplars · {sum(e['trace_id'] in trace_ids for e in ex)} con traza")
    except Exception as e: add("R6", "Exemplars", False, type(e).__name__)
    # R7 ---------------------------------------------------------------------------------------------------
    def window_ok(q):
        ws = re.findall(r"\[(\d+)([smh])\]", q); return bool(ws) and all((int(n) * {"s": 1, "m": 60, "h": 3600}[u]) >= 60 for n, u in ws) or "$__rate_interval" in q
    def names_ok(q):
        toks = set(re.findall(r"proj09_\w+", q)); return bool(toks) and toks <= VALID
    try:
        q = PROMQL
        chk = {"rate": names_ok(q["rate"]) and "rate(" in q["rate"] and window_ok(q["rate"]) and "proj09_agent_requests_total" in q["rate"],
               "error_ratio": names_ok(q["error_ratio"]) and "/" in q["error_ratio"] and "error" in q["error_ratio"] and window_ok(q["error_ratio"]),
               "p95": names_ok(q["p95"]) and "histogram_quantile(0.95" in q["p95"].replace(" ", "") and "sum by (le)" in q["p95"].replace("sum by(le)", "sum by (le)") and "_bucket" in q["p95"] and "rate(" in q["p95"] and window_ok(q["p95"]),
               "cost_per_hour": names_ok(q["cost_per_hour"]) and "proj09_llm_cost_USD_total" in q["cost_per_hour"] and "3600" in q["cost_per_hour"] and "rate(" in q["cost_per_hour"],
               "tokens_per_request": names_ok(q["tokens_per_request"]) and "proj09_llm_input_tokens_total" in q["tokens_per_request"] and "proj09_agent_requests_total" in q["tokens_per_request"] and "/" in q["tokens_per_request"] and window_ok(q["tokens_per_request"]),
               "slowest_tool": names_ok(q["slowest_tool"]) and "topk(1" in q["slowest_tool"].replace(" ", "") and "_sum" in q["slowest_tool"] and "_count" in q["slowest_tool"] and "tool" in q["slowest_tool"]}
        add("R7", "PromQL (estático): 6 consultas con forma, nombres y ventanas válidos", all(chk.values()), "fallan: " + ", ".join(k for k, v in chk.items() if not v) if not all(chk.values()) else "6/6")
    except Exception as e: add("R7", "PromQL", False, type(e).__name__ + ": " + str(e)[:60])
    # L1-L3 (informativas) ---------------------------------------------------------------------------------
    if LIVE_RUN is None:
        for r_, d_ in (("L1", "Consultas devuelven datos en Prometheus"), ("L2", "Valores cercanos a la verdad"), ("L3", "Exemplars en Prometheus")): add(r_, d_ + " (con stack)", None, "sin stack o sin ejecutar el Paso 7")
    else:
        try:
            res_l, t0, t1 = LIVE_RUN; at = t1 + 20; got = {k: prom_instant(v, at=at) for k, v in PROMQL.items()}
            nonempty = {k: bool(v) and v[0]["value"] is not None for k, v in got.items()}
            add("L1", "Las 6 consultas devuelven datos en Prometheus", all(nonempty.values()), str({k: (round(v[0]["value"], 3) if nonempty[k] else None) for k, v in got.items() if k != "slowest_tool"}))
            win = [r for r in res_l if t1 - 40 <= r.start_s <= t1]; n = len(win)
            tr_ = {"rate": n / 60, "error_ratio": np.mean([r.status in ("error", "max_steps") for r in win]), "cost_per_hour": sum(r.cost_usd for r in win) / 60 * 3600,
                   "tokens_per_request": sum(r.input_tokens for r in win) / n}
            rel = {k: abs(got[k][0]["value"] - tr_[k]) / max(abs(tr_[k]), 1e-9) for k in ("rate", "cost_per_hour", "tokens_per_request") if nonempty[k]}
            ok2 = all(v <= 0.45 for v in rel.values()) and nonempty["error_ratio"] and abs(got["error_ratio"][0]["value"] - tr_["error_ratio"]) <= 0.10
            add("L2", "Valores cercanos a la verdad (tolerancias amplias por el desfase del scrape)", ok2, str({k: round(v, 2) for k, v in rel.items()}))
            ex = prom_exemplars("proj09_agent_duration_seconds_bucket", minutes=4)
            add("L3", "Hay exemplars de tu métrica en Prometheus", len(ex) > 0, f"{len(ex)} exemplars")
        except Exception as e:
            for r_ in ("L1", "L2", "L3"): add(r_, "(en vivo)", False, type(e).__name__ + ": " + str(e)[:70])
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Exponencial:** añade una *View* que convierta `proj09.llm.call.duration` en histograma exponencial y envíalo por la **ruta B** (OTLP directo a Prometheus). Compara series y error de p99 con la versión clásica (sección 9 de la lección).
2. **Guardia de cardinalidad:** haz que `Recorder` cuente los valores distintos de `intent` y, si pasan de 8, los agrupe en `"other"`. ¿Cómo lo probarías con una intención nueva inyectada?
3. **Coste «al consultar»:** exporta un *gauge* observable `proj09.price` con atributos `model` y `direction` (`input`/`output`) y reescribe `cost_per_hour` en PromQL multiplicando tokens por precio con `on(model) group_left`. ¿Qué pasa al cambiar un precio?
4. **Presupuesto en CI:** escribe un test que falle si el número real de series de `Recorder` supera `TOTAL_BUDGET` (por ejemplo tras añadir un atributo).
5. **Detector de buckets malos:** dado un histograma exportado (`metric_rows`), calcula qué fracción de observaciones cae en el bucket de mayor masa y alerta si supera el 60 %.

## Reflexión (producción e hilo conductor)

- Tu `cost()` lanza `KeyError` en un modelo desconocido, pero `MetricsInstrumentor` de la lección **no**: cuenta 0 y anota `unknown_models`. ¿Qué es mejor para un sistema en producción que no debe romper el agente por un fallo de telemetría? ¿Cómo **avisarías** sin romper?
- ¿Qué cambia en tu diseño si mañana el agente usa 5 modelos y 12 herramientas? Recalcula el presupuesto de `tool_duration` y `duration`.
- ¿Qué pregunta de finanzas **no** puedes responder con estas métricas (p. ej. «¿cuánto costó el cliente C-0007?»)? ¿Dónde vive esa respuesta (módulos 07-08, 13)?
- En el módulo 10 construirás con estas consultas el dashboard «Salud del agente»: ¿cuáles pondrías en la fila superior y por qué?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine los TODO y vuelve a pasar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
# TODO 1
PRICES = {"mock-small": (0.25, 1.25), "mock-large": (3.00, 15.00)}      # USD por millón de tokens (ILUSTRATIVOS)
def cost(model: str, input_tokens: int, output_tokens: int) -> float:
    pin, pout = PRICES[model]                                           # KeyError si el modelo no está en la tabla
    return (input_tokens * pin + output_tokens * pout) / 1_000_000

# TODO 2
LATENCY_BOUNDS = [0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 2.5, 3, 4, 5, 7.5, 10, 20, 40]
TOKEN_BOUNDS = [100, 200, 300, 350, 400, 450, 500, 600, 800, 1000, 1500, 2000, 4000, 8000]

# TODO 3
def build_metrics(meter) -> dict:
    return {
        "requests": meter.create_counter("proj09.agent.requests", unit="{request}", description="Peticiones por estado final"),
        "duration": meter.create_histogram("proj09.agent.duration", unit="s", description="Duración del agente", explicit_bucket_boundaries_advisory=LATENCY_BOUNDS),
        "in_tokens": meter.create_counter("proj09.llm.input_tokens", unit="{token}", description="Tokens de entrada"),
        "out_tokens": meter.create_counter("proj09.llm.output_tokens", unit="{token}", description="Tokens de salida"),
        "cost": meter.create_counter("proj09.llm.cost", unit="USD", description="Coste (precios ilustrativos)"),
        "llm_duration": meter.create_histogram("proj09.llm.call.duration", unit="s", description="Duración de la llamada al LLM", explicit_bucket_boundaries_advisory=LATENCY_BOUNDS),
        "tool_duration": meter.create_histogram("proj09.tool.duration", unit="s", description="Duración de una herramienta", explicit_bucket_boundaries_advisory=LATENCY_BOUNDS),
        "tokens_per_call": meter.create_histogram("proj09.llm.input_tokens_per_call", unit="{token}", description="Tokens de entrada por llamada", explicit_bucket_boundaries_advisory=TOKEN_BOUNDS),
    }

# TODO 4
class Recorder(Instrumentor):
    def __init__(self, meter_provider):
        self.m = build_metrics(meter_provider.get_meter("proj09"))

    @contextmanager
    def agent_run(self, ctx):
        try:
            yield
        finally:
            r = ctx.result
            if r is not None:
                self.m["requests"].add(1, {"status": r.status, "intent": r.intent_pred or "unknown", "tier": ctx.ticket.tier})
                self.m["duration"].record(r.latency_s, {"model": r.model, "status": r.status})

    @contextmanager
    def llm_call(self, ctx, info):
        t0 = ctx.clock.now_s(); outcome = "ok"
        try:
            yield
        except BaseException:
            outcome = "error"; raise
        finally:
            self.m["llm_duration"].record(max(ctx.clock.now_s() - t0, 0.0), {"model": info.model, "outcome": outcome})
            r = info.response
            if r is not None:
                at = {"model": info.model}
                self.m["in_tokens"].add(r.usage.input_tokens, at); self.m["out_tokens"].add(r.usage.output_tokens, at)
                self.m["cost"].add(cost(info.model, r.usage.input_tokens, r.usage.output_tokens), at)
                self.m["tokens_per_call"].record(r.usage.input_tokens, at)

    @contextmanager
    def tool_call(self, ctx, info):
        t0 = ctx.clock.now_s(); outcome = "ok"
        try:
            yield
        except BaseException:
            outcome = "error"; raise
        finally:
            self.m["tool_duration"].record(max(ctx.clock.now_s() - t0, 0.0), {"tool": info.call.name, "outcome": outcome})

# TODO 5
def series_budget(attrs: dict, n_bounds: int | None = None) -> int:
    return math.prod(attrs.values()) * (n_bounds + 3 if n_bounds is not None else 1)

DECLARED = {
    "requests": series_budget({"status": 5, "intent": 5, "tier": 2}),
    "duration": series_budget({"model": 2, "status": 5}, len(LATENCY_BOUNDS)),
    "in_tokens": series_budget({"model": 2}), "out_tokens": series_budget({"model": 2}), "cost": series_budget({"model": 2}),
    "llm_duration": series_budget({"model": 2, "outcome": 2}, len(LATENCY_BOUNDS)),
    "tool_duration": series_budget({"tool": 4, "outcome": 2}, len(LATENCY_BOUNDS)),
    "tokens_per_call": series_budget({"model": 2}, len(TOKEN_BOUNDS)),
}
TOTAL_BUDGET = sum(DECLARED.values())

# TODO 6
PROMQL = {
    "rate": "sum(rate(proj09_agent_requests_total[1m]))",
    "error_ratio": 'sum(rate(proj09_agent_requests_total{status=~"error|max_steps"}[1m])) / sum(rate(proj09_agent_requests_total[1m]))',
    "p95": "histogram_quantile(0.95, sum by (le) (rate(proj09_agent_duration_seconds_bucket[1m])))",
    "cost_per_hour": "sum(rate(proj09_llm_cost_USD_total[1m])) * 3600",
    "tokens_per_request": "sum(rate(proj09_llm_input_tokens_total[1m])) / sum(rate(proj09_agent_requests_total[1m]))",
    "slowest_tool": "topk(1, sum by (tool) (rate(proj09_tool_duration_seconds_sum[1m])) / sum by (tool) (rate(proj09_tool_duration_seconds_count[1m])))",
}
print("Presupuesto declarado por instrumento:", DECLARED, "· total", TOTAL_BUDGET)
ROWS, RES, TRACE_IDS = run_inmemory()
if LIVE and LIVE_RUN is None:
    LIVE_RUN = run_live_project(60)
display(run_rubric())

In [ ]:
# Visualización de la solución: error de percentil con los límites de la spec frente a los tuyos, y series por instrumento
fig, axes = plt.subplots(1, 3, figsize=(16, 4.1))
lat7, tok7 = sample_runs(7, 1000)
labels = ["p50", "p95", "p99"]; spec = quantile_errors(lat7, SPEC_AGENT_DURATION_BOUNDARIES); mine = quantile_errors(lat7, LATENCY_BOUNDS)
x = np.arange(3); axes[0].bar(x - .2, [abs(spec[k]) for k in labels], .4, color=viz.PALETTE["warn"], label="límites de la spec"); axes[0].bar(x + .2, [abs(mine[k]) for k in labels], .4, color=viz.PALETTE["ok"], label="tus límites")
axes[0].set_xticks(x); axes[0].set_xticklabels(labels); axes[0].set_ylabel("|error| %"); axes[0].set_title("Latencia: error de percentil (semilla 7)"); axes[0].legend()
specT = quantile_errors(tok7, SPEC_TOKEN_BOUNDARIES, (.5, .95)); mineT = quantile_errors(tok7, TOKEN_BOUNDS, (.5, .95))
axes[1].bar([-.2, .8], [abs(specT["p50"]), abs(specT["p95"])], .4, color=viz.PALETTE["warn"]); axes[1].bar([.2, 1.2], [abs(mineT["p50"]), abs(mineT["p95"])], .4, color=viz.PALETTE["ok"])
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["p50", "p95"]); axes[1].set_title("Tokens de entrada: error de percentil (semilla 7)")
axes[2].barh(list(DECLARED), list(DECLARED.values()), color=viz.PALETTE["metrics"]); axes[2].set_title(f"Presupuesto de series ({TOTAL_BUDGET} en total)"); axes[2].set_xlabel("series máx. en Prometheus")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **Panel izquierdo y central:** las barras verdes (tus límites) deberían quedar **por debajo** de las naranjas (los de la spec) en p95/p99: ahí está el valor de poner los *buckets* donde está la masa. Si no, tus límites están demasiado repartidos o dejan un cubo enorme alrededor del p95.
> - **Panel derecho:** qué instrumentos dominan el presupuesto de series. Los histogramas multiplican por `n_bounds + 3`: añadir un atributo a un histograma cuesta mucho más que a un contador.